# **第2章 Python数理最適化チュートリアル**

## **2.1 連立一次方程式をPythonの数理最適化ライブラリで解く**

### 全体のコード

In [ ]:
import pulp

problem = pulp.LpProblem('SLE', pulp.LpMaximize)

x = problem.add_variable('x', cat='Continuous')
y = problem.add_variable('y', cat='Continuous')

problem += 120 * x + 150 * y == 1440
problem += x + y == 10

stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))

print('Status:', stats.status_str)
print('x=', x.value(), 'y=', y.value())

### 本書における逐次実行

In [ ]:
# PythonライブラリPuLPの取り込み
import pulp

In [ ]:
# 数理モデルの定義
problem = pulp.LpProblem('SLE', pulp.LpMaximize)
problem

In [ ]:
# 変数の定義
x = problem.add_variable('x', cat='Continuous')
y = problem.add_variable('y', cat='Continuous')

In [ ]:
# 制約式の定義
problem += 120 * x + 150 * y == 1440
problem += x + y == 10
problem

In [ ]:
# 求解
stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))
print('Status:', stats.status_str)

In [ ]:
# 最適化結果の表示
print('x=', x.value(), 'y=', y.value())

## **2.2 線形計画問題をPythonの数理最適化ライブラリで解く**

### 全体のコード

In [ ]:
import pulp

problem = pulp.LpProblem('LP', pulp.LpMaximize)

x = problem.add_variable('x', cat='Continuous')
y = problem.add_variable('y', cat='Continuous')

problem += 1 * x + 3 * y <= 30
problem += 2 * x + 1 * y <= 40
problem += x >= 0
problem += y >= 0
problem.objective = x + 2 * y

stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))

print('Status:', stats.status_str)
print('x=', x.value(), 'y=', y.value(), 'obj=', problem.objective.value())

### 本書における逐次実行

In [ ]:
# PythonライブラリPuLPの取り込み
import pulp

In [ ]:
# 数理最適化モデルの定義
problem = pulp.LpProblem('LP', pulp.LpMaximize)
problem

In [ ]:
# 変数の定義
x = problem.add_variable('x', cat='Continuous')
y = problem.add_variable('y', cat='Continuous')

In [ ]:
# 制約式の定義
problem += 1 * x + 3 * y <= 30
problem += 2 * x + 1 * y <= 40
problem += x >= 0
problem += y >= 0

In [ ]:
# 目的関数の定義
problem.objective = x + 2 * y
problem 

In [ ]:
# 求解
stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))
print('Status:', stats.status_str)

In [ ]:
# 最適化結果の表示
print('x=', x.value(), 'y=', y.value(), 'obj=', problem.objective.value())

## **2.3 規模の大きな数理最適化問題をPythonの数理最適化ライブラリで解く**

### **線形計画問題**

### ①データのインポート

In [ ]:
# データ処理のためのライブラリpandasとPythonライブラリPuLPの取り込み
import pandas as pd
import pulp

In [ ]:
# stocks.csvからのデータ取得
stock_df = pd.read_csv('stocks.csv')
stock_df

In [ ]:
# requires.csvからのデータ取得
require_df = pd.read_csv('requires.csv')
require_df

In [ ]:
# gains.csvからのデータ取得
gain_df = pd.read_csv('gains.csv')
gain_df

### ②リストの定義

In [ ]:
# 製品のリストの定義
P = gain_df['p'].tolist()
P

In [ ]:
# 原料のリストの定義
M = stock_df['m'].tolist()
M

### ③定数の定義

In [ ]:
# 定数の定義:stock
stock = {row.m:row.stock for row in stock_df.itertuples()}

#stock = dict(zip(stock_df['m'], stock_df['stock']))
#stock = dict((row.m, row.stock) for row in stock_df.itertuples())
#stock = {row['m']:row['stock'] for i, row in stock_df.iterrows()} # 追記:iterrowsは低速なので避ける
#stock = stock_df.set_index('m').to_dict()['stock']
#stock = stock_df.set_index('m')['stock'].to_dict() # 追記
stock

In [ ]:
# 定数の定義:gain
gain = {row.p:row.gain for row in gain_df.itertuples()}
gain

In [ ]:
# 定数の定義:require
require = {(row.p,row.m):row.require for row in require_df.itertuples()}
require

### ④線形計画問題の定義

In [ ]:
# 数理最適化モデルの定義
problem = pulp.LpProblem('LP2', pulp.LpMaximize)

### ⑤変数の定義

In [ ]:
# 変数の定義
x = problem.add_variable_dicts('x', P, cat='Continuous')

# 変数の逐次定義
#x = {}
#for p in P:
#    x[p] = problem.add_variable('x_{}'.format(p), cat='Continuous')

# f-strings(Python3.6以降)
#x = {}
#for p in P:
#    x[p] = problem.add_variable(f'x_{p}', cat='Continuous')    

# 辞書 & f-strings
# x = {p:problem.add_variable(f'x_{p}', cat='Continuous') for p in P}

### ⑥制約式の定義

In [ ]:
# 生産量は0以上
for p in P:
    problem += x[p] >= 0

In [ ]:
# 生産量は在庫の範囲
for m in M:
    problem += pulp.lpSum([require[p,m] * x[p] for p in P]) <= stock[m]

### ⑦目的関数の定義

In [ ]:
# 目的関数の定義    
problem += pulp.lpSum([gain[p] * x[p] for p in P])
problem # 追記

### 実行

In [ ]:
# 求解
stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))
print('Status:', stats.status_str)

In [ ]:
# 計算結果の表示
for p in P:
    print(p, x[p].value())

print('obj=', problem.objective.value())

### ⑧実装した数理最適化モデルのまとめ

In [ ]:
import pandas as pd
import pulp

# データの取得
require_df = pd.read_csv('requires.csv')
stock_df = pd.read_csv('stocks.csv')
gain_df = pd.read_csv('gains.csv')

# 集合の定義
P = gain_df['p'].tolist()
M = stock_df['m'].tolist()

# 定数の定義
stock = {row.m:row.stock for row in stock_df.itertuples()}
gain = {row.p:row.gain for row in gain_df.itertuples()}
require = {(row.p,row.m):row.require for row in require_df.itertuples()}

# 数理最適化モデルの定義
problem = pulp.LpProblem('LP2', pulp.LpMaximize)

# 変数の定義
x = problem.add_variable_dicts('x', P, cat='Continuous')

# 制約式の定義
for p in P:
    problem += x[p] >= 0
for m in M:
    problem += pulp.lpSum([require[p,m] * x[p] for p in P]) <= stock[m]

# 目的関数の定義    
problem += pulp.lpSum([gain[p] * x[p] for p in P])

# 求解
stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))
print('Status:', stats.status_str)

# 計算結果の表示
for p in P:
    print(p, x[p].value())

print('obj=', problem.objective.value())

### **整数計画問題**

### コード全体

In [ ]:
import pandas as pd
import pulp

# データの取得
require_df = pd.read_csv('requires.csv')
stock_df = pd.read_csv('stocks.csv')
gain_df = pd.read_csv('gains.csv')

# 集合の定義
P = gain_df['p'].tolist()
M = stock_df['m'].tolist()

# 定数の定義
stock = {row.m:row.stock for row in stock_df.itertuples()}
gain = {row.p:row.gain for row in gain_df.itertuples()}
require = {(row.p,row.m):row.require for row in require_df.itertuples()}

# 数理最適化モデルの定義
problem = pulp.LpProblem('IP', pulp.LpMaximize) # 変更点（任意）

# 変数の定義
x = problem.add_variable_dicts('x', P, cat='Integer') # 変更点

# 制約式の定義
for p in P:
    problem += x[p] >= 0
for m in M:
    problem += pulp.lpSum([require[p,m] * x[p] for p in P]) <= stock[m]

# 目的関数の定義    
problem += pulp.lpSum([gain[p] * x[p] for p in P])

# 求解
stats = problem.solve(pulp.COIN_CMD(gapRel=0, gapAbs=0))
print('Status:', stats.status_str)

# 計算結果の表示
for p in P:
    print(p, x[p].value())

print('obj=', problem.objective.value())